<div style="background: linear-gradient(135deg, #1A1226 0%, #2D1B3D 100%); padding: 28px 32px; border-radius: 12px; font-family: system-ui, sans-serif;">
<h1 style="color: #F0C040; font-size: 2.2em; margin: 0; letter-spacing: 1px;">📊 Group Project — Instructor Dashboard</h1>
<h2 style="color: #ffffff; font-size: 1.05em; margin: 8px 0 0 0; font-weight: normal;">SCTC 1013 · Elements of Data Science · Spring 2026</h2>
<p style="color: #B0A8B9; margin: 10px 0 0 0; font-size: 0.85em;">Live view of all group submissions · auto-refreshes every 20 s, or click Refresh manually.</p>
</div>


### ⚙️ Configuration

In [ ]:
# Must match student notebook
SUBMISSIONS_FILE = '/home/jovyan/shared-readwrite/group_project/submissions.json'

CHERRY = '#9E1B34'
GOLD   = '#F0C040'
DARK   = '#1A1226'

DOMAIN_COLORS = {
    'DS17': '#4A90D9', 'DS16': '#4A90D9', 'DS20': '#4A90D9',  # Bioinformatics
    'DS1' : '#2E7D50',                                          # Ecology
    'DS2' : '#9E1B34', 'DS9' : '#9E1B34', 'DS10': '#9E1B34',  # Public Health
    'DS3' : '#8B5CF6',                                          # Chemistry
    'DS5' : '#059669', 'DS6' : '#059669', 'DS7' : '#059669',  # Env Sci
    'DS8' : '#059669', 'DS14': '#059669', 'CAMELS': '#059669',
    'DS11': '#D97706', 'DS12': '#D97706', 'DS18': '#D97706',  # General
}

def get_color(dataset_str):
    for key, color in DOMAIN_COLORS.items():
        if key in dataset_str:
            return color
    return '#6B7280'

# Instructor creates the shared directory on first run
import os
os.makedirs(os.path.dirname(SUBMISSIONS_FILE), exist_ok=True)
print('Config loaded — reading from:', SUBMISSIONS_FILE)

---
### Setup

In [ ]:
import json, os, time, threading
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print('Ready')

---
### Dashboard renderer

In [ ]:

def load_submissions():
    try:
        with open(SUBMISSIONS_FILE) as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return []


def render_summary_bar(subs):
    n = len(subs)
    datasets_used = len({s.get('dataset','') for s in subs})
    members_total = sum(len(s.get('members', [])) for s in subs)
    
    stats = [
        ('🏷️', str(n), 'Groups submitted'),
        ('📂', str(datasets_used), 'Unique datasets'),
        ('👥', str(members_total), 'Total members'),
    ]
    cards = ''.join(
        f'<div style="flex:1;min-width:120px;background:#F7F5F0;border-radius:10px;'
        f'padding:14px;text-align:center;">'
        f'<div style="font-size:1.4em;">{icon}</div>'
        f'<div style="font-size:2em;font-weight:800;color:#1A1226;margin:4px 0 2px;">{val}</div>'
        f'<div style="font-size:11px;color:#888;">{label}</div>'
        f'</div>'
        for icon, val, label in stats
    )
    return f'<div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:20px;">{cards}</div>'


def render_group_card(sub):
    team   = sub.get('team', 'Unknown')
    title  = sub.get('title', '—')
    ds     = sub.get('dataset', '—')
    hyp    = sub.get('hypothesis', '')
    desc   = sub.get('description', '')
    members = sub.get('members', [])
    ts     = sub.get('submitted_at', '')
    
    color = get_color(ds)
    
    member_chips = ''.join(
        f'<span style="display:inline-block;background:#F0EDE8;border-radius:20px;'
        f'padding:3px 10px;font-size:11px;margin:2px;color:#333;">'
        f'<strong>{m["name"]}</strong> '
        f'<span style="color:#888;">· {m["role"]}</span></span>'
        for m in members
    )
    
    return f'''
    <div style="background:#fff;border:1px solid #E0DDD6;border-radius:10px;
                margin-bottom:14px;overflow:hidden;box-shadow:0 2px 8px rgba(0,0,0,0.05);">
      <!-- Card header -->
      <div style="background:{color};padding:10px 16px;display:flex;align-items:center;gap:10px;">
        <span style="font-size:1.3em;font-weight:800;color:#fff;letter-spacing:0.5px;">{team}</span>
        <span style="margin-left:auto;font-size:11px;color:rgba(255,255,255,0.75);">{ts}</span>
      </div>
      <!-- Card body -->
      <div style="padding:14px 18px;">
        <div style="font-size:15px;font-weight:700;color:#1A1226;margin-bottom:2px;">{title}</div>
        <div style="font-size:12px;color:#888;margin-bottom:10px;">📂 {ds}</div>
        
        <div style="background:#FDF8F0;border-left:3px solid {color};border-radius:4px;
                    padding:8px 12px;margin-bottom:10px;">
          <div style="font-size:10px;font-weight:700;color:{color};letter-spacing:0.5px;margin-bottom:3px;">HYPOTHESIS</div>
          <div style="font-size:12.5px;color:#333;font-style:italic;">{hyp}</div>
        </div>
        
        <div style="font-size:12px;color:#555;margin-bottom:10px;">{desc}</div>
        
        <div style="border-top:1px solid #F0EDE8;padding-top:8px;">
          <span style="font-size:11px;font-weight:600;color:#888;">TEAM:  </span>
          {member_chips}
        </div>
      </div>
    </div>
    '''


def render_dataset_coverage(subs):
    """Show which datasets are claimed so far."""
    claimed = {}
    for s in subs:
        ds = s.get('dataset', '')
        claimed[ds] = claimed.get(ds, []) + [s.get('team', '?')]
    
    if not claimed:
        return ''
    
    rows = ''.join(
        f'<tr style="border-bottom:1px solid #F0EDE8;">'
        f'<td style="padding:5px 10px;font-size:12px;color:#444;">{ds}</td>'
        f'<td style="padding:5px 10px;font-size:12px;color:#9E1B34;">' +
        ', '.join(teams) + '</td></tr>'
        for ds, teams in sorted(claimed.items())
    )
    return f'''
    <div style="background:#fff;border:1px solid #E0DDD6;border-radius:10px;
                margin-bottom:16px;overflow:hidden;">
      <div style="padding:10px 16px;border-bottom:1px solid #F0EDE8;background:#F7F5F0;">
        <strong>📂 Dataset Coverage</strong>
        <span style="font-size:11px;color:#aaa;margin-left:8px;">which groups claimed which dataset</span>
      </div>
      <table style="width:100%;border-collapse:collapse;">
        <thead><tr style="background:#F7F5F0;">
          <th style="padding:5px 10px;text-align:left;font-size:11px;color:#888;">Dataset</th>
          <th style="padding:5px 10px;text-align:left;font-size:11px;color:#888;">Group(s)</th>
        </tr></thead>
        <tbody>{rows}</tbody>
      </table>
    </div>
    '''


def render_all(output_widget):
    subs = load_submissions()
    ts   = time.strftime('%H:%M:%S')
    
    parts = [
        render_summary_bar(subs),
        f'<div style="font-size:11px;color:#aaa;margin-bottom:16px;">Last refreshed: {ts} · {len(subs)} group(s) submitted</div>',
    ]
    
    if subs:
        parts.append(render_dataset_coverage(subs))
        parts.append('<div style="font-size:14px;font-weight:700;color:#1A1226;margin-bottom:10px;">📋 Group Proposals</div>')
        for sub in sorted(subs, key=lambda x: x.get('timestamp', '')):
            parts.append(render_group_card(sub))
    else:
        parts.append(
            '<div style="color:#888;font-style:italic;padding:20px;text-align:center;">'
            'No submissions yet. Groups should run the <strong>GroupProject_Submission</strong> notebook.</div>'
        )
    
    with output_widget:
        clear_output(wait=True)
        display(HTML('\n'.join(parts)))

print('Renderer ready')


---
### 🚀 Launch Dashboard

In [ ]:

btn_refresh = widgets.Button(
    description='🔄 Refresh Now',
    layout=widgets.Layout(width='160px', height='36px'),
    style={'button_color': CHERRY, 'font_weight': 'bold'}
)
btn_toggle = widgets.Button(
    description='⏸ Pause Auto-Refresh',
    layout=widgets.Layout(width='210px', height='36px')
)
interval_slider = widgets.IntSlider(
    value=20, min=5, max=60, step=5,
    description='Interval (s):',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='320px')
)
status_lbl = widgets.Label(value='Auto-refresh: ON')
output     = widgets.Output()

_auto = [True]
_stop = threading.Event()

def do_refresh(_=None):
    render_all(output)

def toggle_auto(_):
    _auto[0] = not _auto[0]
    if _auto[0]:
        btn_toggle.description = '⏸ Pause Auto-Refresh'
        status_lbl.value = 'Auto-refresh: ON'
    else:
        btn_toggle.description = '▶ Resume Auto-Refresh'
        status_lbl.value = 'Auto-refresh: PAUSED'

btn_refresh.on_click(do_refresh)
btn_toggle.on_click(toggle_auto)

def _loop():
    while not _stop.is_set():
        secs = interval_slider.value
        for _ in range(secs):
            if _stop.is_set():
                return
            import time; time.sleep(1)
        if _auto[0]:
            render_all(output)

_stop.clear()
import threading
threading.Thread(target=_loop, daemon=True).start()

controls = widgets.HBox(
    [btn_refresh, btn_toggle, interval_slider, status_lbl],
    layout=widgets.Layout(gap='12px', align_items='center', margin='0 0 12px 0')
)
display(controls, output)
do_refresh()
print('Dashboard live!')


---
### ✏️ Manual Entry (instructor fallback)

In [ ]:
import tempfile

def manual_submit(team, title, dataset, hypothesis, description, members):
    """
    Manual entry for groups who cannot submit from their notebook.
    members: list of dicts with 'name' and 'role' keys
    
    Example:
    manual_submit(
        team='Broad Street Bootstrappers',
        title='Microbiome Diversity in Health vs. Disease',
        dataset='DS17 · Human Microbiome Diversity',
        hypothesis='Gut microbiome Shannon diversity is lower in IBD patients vs. healthy controls.',
        description='We will use permutation tests to compare diversity indices across health groups.',
        members=[{'name': 'Alice Chen', 'role': 'Coder'}, {'name': 'Bob Kim', 'role': 'Analyst'}]
    )
    """
    entry = {
        'team': team,
        'title': title,
        'dataset': dataset,
        'hypothesis': hypothesis,
        'description': description,
        'members': members,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'submitted_at': time.strftime('%H:%M'),
        'manual': True
    }
    try:
        with open(SUBMISSIONS_FILE) as f:
            subs = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        subs = []
    subs = [s for s in subs if s.get('team', '').lower() != team.lower()]
    subs.append(entry)
    dir_ = os.path.dirname(SUBMISSIONS_FILE)
    os.makedirs(dir_, exist_ok=True)
    with tempfile.NamedTemporaryFile('w', dir=dir_, delete=False, suffix='.tmp') as f:
        json.dump(subs, f, indent=2)
        tmp = f.name
    os.replace(tmp, SUBMISSIONS_FILE)
    print(f'✅ Submitted: {team} · {dataset}')

# Example — uncomment and edit:
# manual_submit(
#     team='Broad Street Bootstrappers',
#     title='Microbiome Diversity in Health vs. Disease',
#     dataset='DS17 · Human Microbiome Diversity',
#     hypothesis='Gut microbiome Shannon diversity is lower in IBD patients vs. healthy controls.',
#     description='We will apply permutation tests to compare diversity indices across health status groups, and bootstrap CIs for the difference in means.',
#     members=[{'name': 'Alice Chen', 'role': 'Coder'}, {'name': 'Bob Kim', 'role': 'Analyst'}]
# )


---
### 🗑️ Reset (wipe all submissions — use with caution)

In [ ]:
# import os
# if os.path.exists(SUBMISSIONS_FILE):
#     os.remove(SUBMISSIONS_FILE)
#     print('Submissions cleared.')